In [1]:
import os, sys, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

WORK = "/kaggle/working"
DATA_DIR = f"{WORK}/data"
OUT_DIR = f"{WORK}/outputs"
RESULTS_DIR = f"{WORK}/results"
for d in (DATA_DIR, OUT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

def find(name):
    hits = sorted(glob.glob(f"{WORK}/**/{name}", recursive=True)) + \
           sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found: attach the dataset/notebook that makes it (Add Input)")
    if len(hits) > 1:
        print(f"note: {len(hits)} copies of {name}, using {hits[0]}")
    return hits[0]

RAW_CSV = find("afri_med_qa_15k_v2.4_phase_2_15275.csv")

HF_USER = "Tresor26"
def hub_id(run_name):
    return f"{HF_USER}/clinicmate-{run_name}"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO = "/kaggle/temp/ClinicMate"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Tresorshingiro/ClinicMate.git {REPO}
else:
    !git -C {REPO} pull -q
sys.path.append(REPO)

import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)

In [2]:
!pip uninstall -q -y torchao
!pip install -q peft bitsandbytes evaluate rouge_score bert_score
import torch; torch.manual_seed(42)
print(torch.cuda.device_count(), torch.cuda.get_device_name(0))   # should print "1 Tesla T4"

import importlib.metadata as md                                   # copy this output into requirements.txt
for p in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "datasets",
          "evaluate", "sentence-transformers", "scikit-learn", "pandas"]:
    print(f"{p}=={md.version(p)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00
1 Tesla T4
torch==2.11.0+cu128
transformers==5.16.1
peft==0.20.0
accelerate==1.14.0
bitsandbytes==0.50.2
datasets==4.8.5
evaluate==0.4.6
sentence-transformers==5.7.0
scikit-learn==1.6.1
pandas==2.3.3


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = hub_id("E2_qwen1.5b_r16")                # your best run
HUB_ID = hub_id("qwen2.5-1.5b")                    # the merged model the Space will load

base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
tok = AutoTokenizer.from_pretrained(BASE)
merged.push_to_hub(HUB_ID, private=True); tok.push_to_hub(HUB_ID, private=True)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Tresor26/clinicmate-qwen2.5-1.5b/commit/d05cda4d0b0e455dd8a4da4086e083a12160bd2c', commit_message='Upload tokenizer', commit_description='', oid='d05cda4d0b0e455dd8a4da4086e083a12160bd2c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Tresor26/clinicmate-qwen2.5-1.5b', endpoint='https://huggingface.co', repo_type='model', repo_id='Tresor26/clinicmate-qwen2.5-1.5b'), pr_revision=None, pr_num=None)

In [4]:
import json, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from src.prompts import build_messages
from src.evaluation import mcq_predict, generate_answers, text_metrics, bootstrap_ci

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

def load(base, adapter=None):
    tok = AutoTokenizer.from_pretrained(base)
    model = AutoModelForCausalLM.from_pretrained(base, quantization_config=bnb, device_map="auto")
    if adapter:
        model = PeftModel.from_pretrained(model, adapter)
    model.eval()
    model.config.use_cache = True
    return model, tok

def run_eval(name, model, tok, retriever=None, eval_file="test_eval",limit=None):
    records = pd.read_json(find(f"{eval_file}.jsonl"), lines=True).to_dict("records")
    if limit: 
        records = records[:limit]
    out, preds_rows = {}, []
    # MCQ: accuracy
    mcq = [r for r in records if r["qtype"] == "mcq"]
    retrieved = [retriever(r) for r in mcq] if retriever else None
    p = mcq_predict(model, tok, mcq, retrieved=retrieved)
    correct = [pi == r["answer_letters"][0] for pi, r in zip(p, mcq)]
    out["mcq_acc"] = float(np.mean(correct)); out["mcq_ci"] = bootstrap_ci(correct)
    for tier in ["expert", "crowdsourced"]:
        c = [ci for ci, r in zip(correct, mcq) if r ["tier"] == tier]
        if c: out[f"mcq_acc_{tier}"] = float(np.mean(c))
    preds_rows += [{**r, "pred": pi, "correct": c} for r, pi, c in zip(mcq, p, correct)]
    # SAQ: generation + ROUGE-L / BERTScore
    sub = [r for r in records if r["qtype"] == "saq"]
    msgs = [ build_messages(r, retriever(r) if retriever else None) for r in sub]
    gen = generate_answers(model, tok, msgs, max_new_tokens=256)
    m = text_metrics(gen, [r["rationale"] for r in sub])
    out["saq"] = {"rougeL": m["rougeL"], "bertscore_f1": m["bertscore_f1"],
                  "avg_words": float(np.mean([len(g.split()) for g in gen]))}
    preds_rows += [{**r, "pred": g, "rougeL": rl, "bertscore": bs}
                  for r, g, rl, bs in zip(sub, gen, m["per_example_rougeL"], m["per_example_bertscore"])]
    json.dump(out, open(f"{RESULTS_DIR}/eval_{name}.json", "w"), indent=2)
    pd.DataFrame(preds_rows).to_json(f"{RESULTS_DIR}/preds_{name}.jsonl", orient="records", lines=True)
    print(name, json.dumps(out, indent=2))
    return out

def evaluate_and_free(name, base, adapter=None, **kw):
    m, t = load(base, adapter); run_eval(name, m, t, **kw)
    del m
    torch.cuda.empty_cache()

In [5]:
merged.eval(); merged.config.use_cache = True
run_eval("E2_merged_fp16", merged, tok)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


E2_merged_fp16 {
  "mcq_acc": 0.5493910690121786,
  "mcq_ci": [
    0.5155615696887687,
    0.5872801082543978
  ],
  "mcq_acc_expert": 0.5434173669467787,
  "mcq_acc_crowdsourced": 0.72,
  "saq": {
    "rougeL": 0.31248312545921186,
    "bertscore_f1": 0.33356006236204594,
    "avg_words": 37.864197530864196
  }
}


{'mcq_acc': 0.5493910690121786,
 'mcq_ci': (0.5155615696887687, 0.5872801082543978),
 'mcq_acc_expert': 0.5434173669467787,
 'mcq_acc_crowdsourced': 0.72,
 'saq': {'rougeL': 0.31248312545921186,
  'bertscore_f1': 0.33356006236204594,
  'avg_words': 37.864197530864196}}

In [6]:
import numpy as np, pandas as pd, torch
from src.prompts import build_messages
from src.evaluation import text_metrics

saq = pd.read_json(find("test_eval.jsonl"), lines=True).query("qtype == 'saq'").to_dict("records")
msgs = [build_messages(r) for r in saq]

@torch.no_grad()
def gen(msgs, bs=8, **kw):
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(msgs), bs):
        texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs[i:i + bs]]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(merged.device)
        out = merged.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id, **kw)
        outs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

def looping(text, n=4):                       # under 70% unique 4-grams = stuck in a loop
    w = text.split(); grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return bool(grams) and len(set(grams)) / len(grams) < 0.7

SETTINGS = {
    "eval (rp 1.1)":        dict(repetition_penalty=1.1),
    "rp 1.1 + no_repeat_4": dict(repetition_penalty=1.1, no_repeat_ngram_size=4),
    "rp 1.2":               dict(repetition_penalty=1.2),
}
rows = []
for name, kw in SETTINGS.items():
    g = gen(msgs, **kw)
    m = text_metrics(g, [r["rationale"] for r in saq])
    rows.append({"setting": name, "rougeL": m["rougeL"], "bertscore": m["bertscore_f1"],
                 "looping_%": 100 * np.mean([looping(x) for x in g]),
                 "avg_words": np.mean([len(x.split()) for x in g])})
table = pd.DataFrame(rows).round(3)
table.to_csv(f"{RESULTS_DIR}/decoding_check.csv", index=False); print(table)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


                setting  rougeL  bertscore  looping_%  avg_words
0         eval (rp 1.1)   0.312      0.334      0.823     37.864
1  rp 1.1 + no_repeat_4   0.284      0.308      0.000     37.823
2                rp 1.2   0.226      0.232      0.000     68.712


Pick the setting with the lowest `looping_%` whose ROUGE-L / BERTScore is not worse than `eval (rp 1.1)`, and put it in `app/app.py`.

In [7]:
from huggingface_hub import ModelCard

card = ModelCard(f"""---
license: cc-by-sa-4.0
base_model: Qwen/Qwen2.5-1.5B-Instruct
language: en
tags: [medical, question-answering, qlora, africa]
---
# ClinicMate: Qwen2.5-1.5B-Instruct fine-tuned on AfriMed-QA

**Student research project. Not for clinical use. Not medical advice.**

- **Base model:** Qwen/Qwen2.5-1.5B-Instruct (Apache-2.0), QLoRA r=16 adapter `{ADAPTER}` merged into fp16 weights.
- **Data:** AfriMed-QA v2 (intronhealth/afrimedqa_v2, CC-BY-SA-4.0), MCQ + SAQ only, own 70/10/20 re-split.
- **Results (our test split):** MCQ accuracy 0.541 (zero-shot base: 0.498); SAQ ROUGE-L 0.292, BERTScore 0.302.
- **Intended use:** demonstrating domain adaptation for medical QA in African contexts, behind a domain gate.

## Known limitations
- Small model; 50% of hand-checked answers contained at least one hallucinated claim, 10% were judged potentially harmful.
- Long list-style answers can fall into repetition loops.
- Much weaker on lay-language patient questions (no such training data with answers).
- English only; question writers are mostly Nigerian.
""")
card.push_to_hub(HUB_ID)


CommitInfo(commit_url='https://huggingface.co/Tresor26/clinicmate-qwen2.5-1.5b/commit/1eadb41ae5a05818b3e8c65a6513af397673444e', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='1eadb41ae5a05818b3e8c65a6513af397673444e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Tresor26/clinicmate-qwen2.5-1.5b', endpoint='https://huggingface.co', repo_type='model', repo_id='Tresor26/clinicmate-qwen2.5-1.5b'), pr_revision=None, pr_num=None)